# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yara-08/ML-FlyRank-Internship2/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

> **Method: Random Forest Classifier**

> I chose a Random Forest Classifier because my lane is Refresh / Content Opportunity Scoring, where the goal is to identify pages that are likely to need a content refresh. The dataset contains many numeric and categorical features that may interact in non-linear ways, so a Random Forest can learn more complex patterns than the simple rule-based baseline from Week 4. It also requires little feature scaling and provides feature importance, making it easier to interpret which signals influence the predictions. I will compare this model against my Week 4 baseline using the same data split and evaluation metrics to determine whether the additional complexity provides a meaningful improvement.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd

# Load dataset
df = pd.read_csv("/content/ML-FlyRank-Internship2/data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

print("\nChosen method:")
print("RandomForestClassifier")

Rows: 30,000
Columns: 44

Chosen method:
RandomForestClassifier


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

> **Split design: Grouped train/test split by client**

> I used a grouped train/test split based on client_id so that pages from the same client do not appear in both the training and test sets. This is a more honest evaluation because content from the same client may share similar characteristics, and allowing those pages to appear in both sets could make the model appear more accurate than it really is. The baseline rule and the Random Forest model will both be evaluated on the same held-out test set against the observed page trend (*trend_direction*) to ensure a fair comparison.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

# Groups = client IDs
groups = df["client_id"]

# Grouped train/test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Training rows: {len(train_df):,}")
print(f"Testing rows: {len(test_df):,}")

print(f"Training clients: {train_df['client_id'].nunique()}")
print(f"Testing clients: {test_df['client_id'].nunique()}")

# Verify no client appears in both sets
shared_clients = (
    set(train_df["client_id"])
    & set(test_df["client_id"])
)

print(f"Shared clients: {len(shared_clients)}")

Training rows: 23,837
Testing rows: 6,163
Training clients: 25
Testing clients: 7
Shared clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

> I trained a Random Forest Classifier and compared it with my Week 4 baseline using the same grouped train/test split and evaluation metrics. The baseline rule was intentionally conservative because it recommended refresh for a limited set of pages based on freshness and visibility. On the held-out test set, the Random Forest achieved higher accuracy, precision, recall, and F1 score than the baseline. Permutation importance showed that recent impression metrics contributed most to the model's predictions. These results suggest that using a broader set of features improved the model's ability to identify declining pages while remaining a decision-support tool rather than evidence of causation.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# -------------------------------
# Target variable
# -------------------------------

# 1 = page is declining
# 0 = page is stable or improving

df["target"] = (df["trend_direction"] == "down").astype(int)

print(df["target"].value_counts())

print("\nDeclining rate:")
print(df["target"].mean().round(3))


# -------------------------------
# Week 4 baseline predictions
# -------------------------------

freshness_points = {
    "0-30": 0,
    "31-90": 1,
    "91-180": 2,
    "181+": 3,
}

visibility_points = {
    "low": 0,
    "moderate": 1,
    "good": 2,
    "excellent": 3,
}

df["baseline_score"] = (
    df["freshness_tier"].map(freshness_points)
    + df["impression_tier"].map(visibility_points)
)

df["baseline_prediction"] = (
    df["baseline_score"] >= 4
).astype(int)

print(df["baseline_prediction"].value_counts())

# -------

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(len(train_df), len(test_df))

# Columns NEVER used as features

drop_columns = [
    "content_id",
    "client_id",

    "trend_direction",
    "trend_pct",

    "target",

    "baseline_score",
    "baseline_prediction"
]

X_train = train_df.drop(columns=drop_columns)
X_test = test_df.drop(columns=drop_columns)

y_train = train_df["target"]
y_test = test_df["target"]


# -------

X_train = pd.get_dummies(
    X_train,
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    drop_first=True
)

# Match columns

X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

print(X_train.shape)


# -------------------------------
# Missing values check
# -------------------------------

missing = X_train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
display(missing)

# Add missing-value indicator columns
missing_cols = [
    "char_count",
    "word_count",
    "competition",
    "search_volume",
    "cpc",
    "scroll_rate",
]

for col in missing_cols:
    X_train[f"{col}_missing"] = X_train[col].isna().astype(int)
    X_test[f"{col}_missing"] = X_test[col].isna().astype(int)


# Fill numeric NaNs with the training median

for col in missing_cols:
    median = X_train[col].median()

    X_train[col] = X_train[col].fillna(median)
    X_test[col] = X_test[col].fillna(median)

print("Remaining missing values (train):", X_train.isna().sum().sum())
print("Remaining missing values (test):", X_test.isna().sum().sum())

target
1    16262
0    13738
Name: count, dtype: int64

Declining rate:
0.542
baseline_prediction
0    26456
1     3544
Name: count, dtype: int64
23837 6163
(23837, 60)
Columns with missing values:


,0
char_count,6614
word_count,6614
competition,2319
search_volume,2319
cpc,2319
scroll_rate,15


Remaining missing values (train): 0
Remaining missing values (test): 0


In [42]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# -------------------------------
# Train Random Forest
# -------------------------------

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
)

rf.fit(X_train, y_train)

rf_predictions = rf.predict(X_test)

# -------------------------------
# Baseline predictions
# -------------------------------

baseline_predictions = test_df["baseline_prediction"]

# -------------------------------
# Comparison table
# -------------------------------

comparison = pd.DataFrame(
    {
        "Method": [
            "Week 4 Baseline",
            "Random Forest",
        ],
        "Accuracy": [
            accuracy_score(y_test, baseline_predictions),
            accuracy_score(y_test, rf_predictions),
        ],
        "Precision": [
            precision_score(y_test, baseline_predictions),
            precision_score(y_test, rf_predictions),
        ],
        "Recall": [
            recall_score(y_test, baseline_predictions),
            recall_score(y_test, rf_predictions),
        ],
        "F1 Score": [
            f1_score(y_test, baseline_predictions),
            f1_score(y_test, rf_predictions),
        ],
    }
).round(3)

display(comparison)

base_rate = y_test.mean()

print(f"Positive class rate in test set: {base_rate:.3f}")

#------

feature_importance = (
    pd.DataFrame(
        {
            "Feature": X_train.columns,
            "Importance": rf.feature_importances_,
        }
    )
    .sort_values(
        "Importance",
        ascending=False,
    )
)

print("Top 10 Features")

display(feature_importance.head(10))


# Permutation Importance

from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="f1"
)

perm_importance = (
    pd.DataFrame(
        {
            "Feature": X_test.columns,
            "Importance": perm.importances_mean,
        }
    )
    .sort_values("Importance", ascending=False)
)

print("Top 10 Permutation Importances")
display(perm_importance.head(10))

,Method,Accuracy,Precision,Recall,F1 Score
0,Week 4 Baseline,0.482,0.399,0.028,0.052
1,Random Forest,0.826,0.824,0.838,0.831


Positive class rate in test set: 0.511
Top 10 Features


,Feature,Importance
18,impressions_prev_30d,0.168725
15,impressions_last_30d,0.132308
5,impressions_90d,0.066959
25,avg_position,0.054267
13,days_with_impressions,0.052331
21,content_age_days,0.038759
17,sessions_last_30d,0.027848
24,ctr,0.024881
16,clicks_last_30d,0.024259
4,char_count,0.023705


Top 10 Permutation Importances


,Feature,Importance
15,impressions_last_30d,0.276184
18,impressions_prev_30d,0.265571
5,impressions_90d,0.018879
16,clicks_last_30d,0.009360
13,days_with_impressions,0.008343
54,impression_tier_low,0.005764
17,sessions_last_30d,0.005591
55,impression_tier_moderate,0.003610
24,ctr,0.002636
6,clicks_90d,0.001672


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

> The Random Forest classified most pages correctly, but it still misclassified **1,074** pages in the test set. The confusion matrix showed both false positives (563) and false negatives (511), indicating that some pages had characteristics that made them difficult to distinguish using the available features. The permutation importance analysis showed that *impressions_last_30d*, *impressions_prev_30d*, and *impressions_90d* were the strongest contributors to the model's predictions, suggesting that recent search visibility was an important signal for identifying declining pages. The misclassified examples showed pages with mixed performance signals, such as low click-through rates but different update histories or search visibility. These results should be viewed as decision-support rather than proof of causation, and refresh decisions may also depend on factors that are not included in this dataset.

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.metrics import confusion_matrix

# -------------------------------
# Confusion Matrix
# -------------------------------

cm = confusion_matrix(y_test, rf_predictions)

print("Confusion Matrix")
print(cm)

# -------------------------------
# Show three misclassified pages
# -------------------------------

results = test_df.copy()

results["Actual"] = y_test.values
results["Predicted"] = rf_predictions

errors = results[
    results["Actual"] != results["Predicted"]
]

print(f"\nTotal misclassified pages: {len(errors)}")

display(
    errors[
        [
            "content_id",
            "client_id",
            "trend_direction",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "days_since_last_update",
            "Actual",
            "Predicted",
        ]
    ].head(3)
)

tn, fp, fn, tp = cm.ravel()

print(f"True Positives : {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Negatives : {tn}")

Confusion Matrix
[[2451  563]
 [ 511 2638]]

Total misclassified pages: 1074


,content_id,client_id,trend_direction,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,Actual,Predicted
1,content_a1fb4e703a9e,client_4e07408562,down,15320,7,0.05,20.3,25,1,0
13,content_a5a2fbc76336,client_8527a891e2,stable,307,0,0.00,39.8,103,0,1
26,content_72c5c2d73e5a,client_4e07408562,stable,2426,3,0.12,30.0,13,0,1


True Positives : 2638
False Positives: 563
False Negatives: 511
True Negatives : 2451


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.